# NovaBank: Predictive Retention at Scale
### Analytics Methods and Frameworks — Project Notebook

**Authors:** Mukabalengu S. Mukuni and Mduduzi Ndlovu
**Date:** May 2026  
**Dataset:** Bank Marketing Dataset — 45,211 customer records  

---

---

## AI Usage Log
This project was developed with assistance from **Claude (Anthropic)** and **ChatGPT**. Specifically:
- - **Feature engineering:** Claude ** suggested splitting `pdays=-1` into a binary flag + numeric value, and proposed duration bucketing and age grouping.
- **Model selection:**ChatGPT** recommended class_weight='balanced' to handle imbalance, and guided the choice of evaluation metrics (AUC + Precision@top20%).
- **Threshold analysis:** Claude ** built the cost-benefit framework using estimated call cost ($8) vs. subscriber value ($350).
- **Segmentation:** Claude ** designed the k-means pipeline and RF score overlay methodology.
- **Code review:**ChatGPT** reviewed all code for data leakage risks, particularly around the train/test split and scaler fitting.

All analytical decisions, interpretations, and final recommendations are our own.

---

## Step 1: Problem Framing

### Problem Brief (5 sentences)
NovaBank is losing revenue as a growing share of customers decline term deposit offers during outbound campaigns, eroding a key source of stable funding. The decision at hand is which customers to proactively target in the next campaign cycle to maximise subscription conversions while minimising wasted outreach. This decision is owned by the VP of Customer Success and the Head of Operations, who control campaign budgets and call-centre capacity. The model must produce actionable scores before each monthly campaign batch is finalised. Any recommendation must be explainable and fair — it cannot discriminate by protected attributes or be a black box that compliance officers cannot interrogate.

### Analytics Problem Statement
Train a binary classification model on historical campaign data to predict the probability that a given customer will subscribe to a term deposit (`y = yes`), and use that score to rank and prioritise outreach — targeting customers above a probability threshold of 0.15.

### Success Metrics (KPIs)
| Metric | Why it matters |
|--------|----------------|
| **AUC-ROC** | Overall model discrimination ability — target > 0.85 |
| **Precision @ top 20%** | Of the customers we call, what % actually subscribe — directly tied to campaign ROI |
| **Recall @ top 20%** | Of all true subscribers, how many do we reach — measures missed revenue |

### Target Action / Policy
> **Contact every customer whose predicted subscription probability exceeds 0.15. Skip the remaining customers. Review threshold monthly against live precision metrics.**

## Step 2: Data Readiness — Load, Profile, Clean, Engineer, Split

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ── LOAD ──────────────────────────────────────────────────────────────────────
# Update path if running locally or in Colab
df = pd.read_csv('bank-full.csv', sep=';')
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Missing values: {df.isnull().sum().sum()}')
df.head()

In [ ]:
# ── DATA DICTIONARY ───────────────────────────────────────────────────────────
data_dict = {
    'age':       'Customer age in years (18-95)',
    'job':       'Occupation type (management, blue-collar, technician, etc.)',
    'marital':   'Marital status — married, single, divorced',
    'education': 'Highest education level — primary, secondary, tertiary',
    'default':   'Has credit in default? yes/no',
    'balance':   'Average yearly account balance in euros (can be negative)',
    'housing':   'Has a housing loan? yes/no',
    'loan':      'Has a personal loan? yes/no',
    'contact':   'Contact method — cellular, telephone, unknown',
    'day':       'Day of month of last contact',
    'month':     'Month of last contact',
    'duration':  'Duration of last call in seconds — longer = more engaged',
    'campaign':  'Number of contacts in this campaign',
    'pdays':     'Days since last contacted from prior campaign (-1 = never)',
    'previous':  'Number of contacts before this campaign',
    'poutcome':  'Outcome of the previous campaign',
    'y':         'TARGET — Did the customer subscribe to a term deposit? yes/no'
}
pd.DataFrame(data_dict.items(), columns=['Column', 'Description'])

### Exploratory Data Analysis (EDA)

Before modeling, we explore how demographic and financial characteristics
relate to subscription behavior. The goal is to identify which customer
attributes signal higher likelihood of converting — informing both feature
selection and the business narrative.



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('EDA: Subscription Behavior by Customer Characteristics',
             fontsize=14, fontweight='bold')

# DEFENSIVE ENCODING: y may still be 'yes'/'no' strings (if this cell runs
# before Step 2) OR already 0/1 integers (if Step 2 ran first).
# We create a fresh y_num column that handles both cases safely.
df_eda = df.copy()
df_eda['y_num'] = df_eda['y'].map({'yes': 1, 'no': 0}) \
                              .fillna(df_eda['y']) \
                              .astype(float)

# 1. Education vs subscription
edu_rate = df_eda.groupby('education')['y_num'].mean().sort_values() * 100
edu_rate.plot(kind='barh', ax=axes[0,0], color='#1D9E75')
axes[0,0].set_title('Subscription rate by education')
axes[0,0].set_xlabel('Subscription rate (%)')

# 2. Marital status vs subscription
mar_rate = df_eda.groupby('marital')['y_num'].mean().sort_values() * 100
mar_rate.plot(kind='barh', ax=axes[0,1], color='#185FA5')
axes[0,1].set_title('Subscription rate by marital status')
axes[0,1].set_xlabel('Subscription rate (%)')

# 3. Balance boxplot — map float 0.0/1.0 to readable labels
df_plot = df_eda.copy()
df_plot['balance_clip'] = df_plot['balance'].clip(-500, 5000)
df_plot['Subscribed'] = df_plot['y_num'].map({0.0: 'No', 1.0: 'Yes'})
df_plot.boxplot(column='balance_clip', by='Subscribed', ax=axes[1,0],
                boxprops=dict(color='#1D9E75'),
                medianprops=dict(color='#185FA5', linewidth=2))
axes[1,0].set_title('Account balance by subscription outcome')
axes[1,0].set_xlabel('Subscribed')
axes[1,0].set_ylabel('Balance (€, clipped at 5K)')
plt.sca(axes[1,0])
plt.title('Account balance by subscription outcome')

# 4. Job category subscription rates
job_rate = df_eda.groupby('job')['y_num'].mean().sort_values() * 100
colors_job = ['#E24B4A' if v < 8 else '#1D9E75' if v > 18 else '#FAC775'
              for v in job_rate.values]
job_rate.plot(kind='barh', ax=axes[1,1], color=colors_job)
axes[1,1].set_title('Subscription rate by job type')
axes[1,1].set_xlabel('Subscription rate (%)')

plt.tight_layout()
plt.show()

### EDA Key Insights

| Finding | Business implication |
|---------|---------------------|
| Tertiary-educated customers subscribe at higher rates | Education is a useful targeting signal |
| Single customers are slightly more responsive | Marital status adds marginal predictive value |
| Subscribers have higher average account balances ($1,804 vs $1,304) | Balance is a meaningful feature — included in our model |
| Students (28.7%) and retirees (22.7%) have the highest job-level subscription rates | Small segments but very high-value — link to Engaged Converters cluster |
| Blue-collar workers convert at only 7% | Lowest-priority segment for term deposit campaigns |


In [ ]:
# ── TARGET DISTRIBUTION ───────────────────────────────────────────────────────
print('Target distribution:')
print(df['y'].value_counts())
print(f'Positive rate: {(df["y"]=="yes").mean():.1%}')
print()
print('NOTE: Dataset is imbalanced (11.7% positive).')
print('Accuracy alone is misleading — always use AUC + precision/recall.')

fig, ax = plt.subplots(figsize=(6, 3))
df['y'].value_counts().plot(kind='bar', ax=ax, color=['#3266ad','#1D9E75'], edgecolor='none')
ax.set_title('Target variable distribution')
ax.set_ylabel('Count')
ax.set_xticklabels(['No subscription', 'Subscribed'], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ── TARGET ENCODING ───────────────────────────────────────────────────────────
# WHY: Models require numeric targets.
df['y'] = (df['y'] == 'yes').astype(int)

# ── FIX pdays = -1 ────────────────────────────────────────────────────────────
# WHY: -1 means 'never contacted before' — a meaningful signal.
# Split into binary flag + numeric value so model uses both signals independently.
df['was_previously_contacted'] = (df['pdays'] != -1).astype(int)
df['pdays'] = df['pdays'].replace(-1, 0)
print(f'Previously contacted: {df["was_previously_contacted"].mean():.1%} of customers')

# ── ENGINEERED FEATURES ───────────────────────────────────────────────────────
# WHY: Raw numbers hide non-linear patterns. Bucketing captures them.

# Call duration bands — subscribers have much longer calls
df['duration_bucket'] = pd.cut(
    df['duration'],
    bins=[0, 60, 180, 360, 600, 9999],
    labels=['very_short','short','medium','long','very_long']
)

# Age groups — young students and older retirees subscribe at 2-4x average rate
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 25, 35, 45, 55, 65, 100],
    labels=['under25','25-34','35-44','45-54','55-64','65+']
)

# Negative balance flag — may signal financial stress
df['negative_balance'] = (df['balance'] < 0).astype(int)

# High campaign contact — over-contacted customers disengage
df['high_campaign_count'] = (df['campaign'] > 5).astype(int)

print('Engineered features added: was_previously_contacted, duration_bucket,',
      'age_group, negative_balance, high_campaign_count')

In [ ]:
# ── CATEGORICAL ENCODING ──────────────────────────────────────────────────────

# Ordinal: education has a natural order
edu_order = {'unknown': 0, 'primary': 1, 'secondary': 2, 'tertiary': 3}
df['education_ord'] = df['education'].map(edu_order)

# Binary yes/no columns
for col in ['default', 'housing', 'loan']:
    df[col] = (df[col] == 'yes').astype(int)

# One-hot encode nominal categoricals
# WHY: drop_first=True avoids multicollinearity (dummy variable trap)
nominal_cols = ['job', 'marital', 'contact', 'month', 'poutcome',
                'duration_bucket', 'age_group']
df_encoded = pd.get_dummies(df, columns=nominal_cols, drop_first=True)
df_encoded = df_encoded.drop(columns=['education'])  # replaced by ordinal

print(f'After encoding: {df_encoded.shape[1]} columns')

# ── TRAIN / TEST SPLIT ────────────────────────────────────────────────────────
# WHY: 80/20 split. Stratified so both sets preserve the 11.7% positive rate.
X = df_encoded.drop(columns=['y'])
y = df_encoded['y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]:,} rows | positive rate: {y_train.mean():.1%}')
print(f'Test : {X_test.shape[0]:,} rows  | positive rate: {y_test.mean():.1%}')

# ── SCALE NUMERIC FEATURES ────────────────────────────────────────────────────
# WHY: Logistic regression is sensitive to feature scale.
# CRITICAL: Fit scaler on TRAIN only — applying to test prevents data leakage.
numeric_cols = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']
scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])

print('Scaling complete. Data is ready for modeling.')

## Step 3: Baseline Model — Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, roc_auc_score,
                             confusion_matrix, average_precision_score,
                             ConfusionMatrixDisplay)

# ── NAIVE BASELINE: always predict 'no' ───────────────────────────────────────
# WHY: Demonstrates that high accuracy alone is meaningless on imbalanced data.
naive_acc = 1 - y_test.mean()
print(f'Naive baseline accuracy (always predict no): {naive_acc:.1%}')
print('  → Looks good but catches ZERO subscribers. Useless in practice.')
print()

# ── LOGISTIC REGRESSION ───────────────────────────────────────────────────────
# class_weight='balanced' compensates for the 88/12 class imbalance
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
lr_prob = lr.predict_proba(X_test)[:, 1]

auc      = roc_auc_score(y_test, lr_prob)
avg_prec = average_precision_score(y_test, lr_prob)

print(f'AUC-ROC       : {auc:.4f}')
print(f'Avg Precision : {avg_prec:.4f}')
print()
print(classification_report(y_test, lr_pred, target_names=['No','Yes']))

In [ ]:
# ── PRECISION @ TOP 20% ───────────────────────────────────────────────────────
def top20_stats(probs, y_true):
    n = int(len(y_true) * 0.20)
    idx = np.argsort(probs)[::-1][:n]
    prec = y_true.iloc[idx].mean()
    rec  = y_true.iloc[idx].sum() / y_true.sum()
    return prec, rec

lr_p20, lr_r20 = top20_stats(lr_prob, y_test)
print(f'Precision @ top 20%: {lr_p20:.1%}')
print(f'Recall    @ top 20%: {lr_r20:.1%}')
print()
print('Interpretation: Targeting the top 20% captures',
      f'{lr_r20:.0%} of all subscribers')
print('vs. 11.7% expected from random calling — a {:.1f}x lift.'.format(lr_p20/0.117))

# ── CONFUSION MATRIX ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, lr_pred, display_labels=['No','Yes'],
    colorbar=False, ax=ax, cmap='Blues'
)
ax.set_title('Logistic Regression — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# ── FEATURE IMPORTANCE (Logistic Regression Coefficients) ────────────────────
coef_df = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': lr.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#1D9E75' if c > 0 else '#E24B4A' for c in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors)
ax.set_title('Logistic Regression — Top 15 Feature Coefficients')
ax.set_xlabel('Coefficient (positive = increases subscription probability)')
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print('Key insight: Call duration is the strongest predictor.')
print('NOTE: Duration is a leaky feature — see Step 4 for handling.')

## Step 4: Improved Model — Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# ── RANDOM FOREST ─────────────────────────────────────────────────────────────
# WHY over logistic regression:
#   - Captures non-linear patterns and feature interactions automatically
#   - No need to manually engineer interaction terms
#   - Provides feature importances for explainability
#   - More robust to outliers in features like balance

rf = RandomForestClassifier(
    n_estimators=300,       # 300 trees — good balance of stability vs. speed
    max_depth=15,           # prevents overfitting on training data
    min_samples_leaf=10,    # each leaf must have at least 10 samples
    class_weight='balanced',# compensates for 88/12 imbalance
    random_state=42,
    n_jobs=-1               # use all CPU cores
)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

rf_auc  = roc_auc_score(y_test, rf_prob)
rf_ap   = average_precision_score(y_test, rf_prob)
rf_p20, rf_r20 = top20_stats(rf_prob, y_test)

print(f'AUC-ROC          : {rf_auc:.4f}')
print(f'Avg Precision    : {rf_ap:.4f}')
print(f'Precision @ top20: {rf_p20:.1%}')
print(f'Recall    @ top20: {rf_r20:.1%}')
print()
print(classification_report(y_test, rf_pred, target_names=['No','Yes']))

In [ ]:
# ── MODEL COMPARISON TABLE ────────────────────────────────────────────────────
results = pd.DataFrame([
    {'Model': 'Naive baseline',      'AUC': 0.500, 'Avg Precision': 0.117, 'Prec@20%': 0.127, 'Recall@20%': 0.216},
    {'Model': 'Logistic Regression', 'AUC': round(roc_auc_score(y_test, lr_prob), 3),
     'Avg Precision': round(average_precision_score(y_test, lr_prob), 3),
     'Prec@20%': round(lr_p20, 3), 'Recall@20%': round(lr_r20, 3)},
    {'Model': 'Random Forest',       'AUC': round(rf_auc, 3),
     'Avg Precision': round(rf_ap, 3),
     'Prec@20%': round(rf_p20, 3), 'Recall@20%': round(rf_r20, 3)},
])
print('=== Baseline vs Improved Model Results ===')
print(results.to_string(index=False))

In [ ]:
# ── FEATURE IMPORTANCE (Random Forest) ───────────────────────────────────────
fi = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(fi['feature'], fi['importance'], color='#185FA5')
ax.set_title('Random Forest — Top 15 Feature Importances')
ax.set_xlabel('Feature importance (% of total predictive power)')
plt.tight_layout()
plt.show()

In [ ]:
# ── DURATION LEAKAGE ANALYSIS ─────────────────────────────────────────────────
# IMPORTANT: Duration is only known AFTER the call. A pre-call scoring model
# cannot use it. We build a second model without duration to simulate
# realistic deployment (Stage 1 of the two-stage system).

dur_cols = [c for c in X_train.columns if 'duration' in c]
X_tr_nd = X_train.drop(columns=dur_cols)
X_te_nd = X_test.drop(columns=dur_cols)

rf_nd = RandomForestClassifier(
    n_estimators=300, max_depth=15, min_samples_leaf=10,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_nd.fit(X_tr_nd, y_train)
nd_prob = rf_nd.predict_proba(X_te_nd)[:, 1]

nd_auc = roc_auc_score(y_test, nd_prob)
nd_p20, nd_r20 = top20_stats(pd.Series(nd_prob), y_test)

print('=== Duration Leakage Comparison ===')
print(f'Full RF (with duration) — AUC: {rf_auc:.3f} | Prec@20%: {rf_p20:.1%} | Recall@20%: {rf_r20:.1%}')
print(f'RF without duration     — AUC: {nd_auc:.3f} | Prec@20%: {nd_p20:.1%} | Recall@20%: {nd_r20:.1%}')
print()
print('Recommendation: Use RF-without-duration for pre-call scoring (Stage 1).')
print('Use full RF after calls to prioritise immediate follow-up (Stage 2).')

### Explainability Framework

For results to be defensible to compliance officers and non-technical
stakeholders, we need more than a good AUC score. We need to explain
*why* the model makes the predictions it does.

**Why Logistic Regression is interpretable:**
Each coefficient directly shows the direction and magnitude of a feature's
effect on subscription probability. A positive coefficient means the feature
increases probability; negative means it decreases it.

**Why Random Forest requires extra explanation effort:**
RF is a black box by default — it averages 300 decision trees. Feature
importance scores tell us *which* features matter but not *how*.
The cell below makes RF explainable by showing the top drivers in plain language.

In [ ]:
# ── PLAIN-LANGUAGE EXPLAINABILITY TABLE ───────────────────────────────────────
# This is what a compliance officer or executive should see — not coefficients,
# but business-readable descriptions of what drives the model.

explainability = pd.DataFrame([
    {'Driver': 'Call duration',
     'Direction': 'Positive',
     'Plain-language explanation': 'Customers who stay on the phone longer are genuinely engaged. '
                                   'A call over 10 minutes makes subscription 4× more likely.',
     'Fairness note': 'No protected attribute — OK to use post-call'},
    {'Driver': 'Previous campaign success',
     'Direction': 'Positive',
     'Plain-language explanation': 'Customers who subscribed in a prior campaign convert at ~65% '
                                   'vs ~8% for those with no prior history.',
     'Fairness note': 'Behavioural — no fairness concern'},
    {'Driver': 'Contact method: unknown',
     'Direction': 'Negative',
     'Plain-language explanation': 'Customers contacted via unknown method are less engaged '
                                   '— likely cold outreach with no relationship.',
     'Fairness note': 'No protected attribute — OK'},
    {'Driver': 'Age group',
     'Direction': 'Non-linear',
     'Plain-language explanation': 'Under-25 and over-65 customers subscribe at 2-4× the '
                                   'rate of 35–54 year olds.',
     'Fairness note': '⚠ Age is a protected attribute — audit required before deployment'},
    {'Driver': 'Housing loan',
     'Direction': 'Negative',
     'Plain-language explanation': 'Customers with housing loans are less likely to subscribe, '
                                   'possibly due to financial commitments reducing disposable income.',
     'Fairness note': 'No protected attribute — OK'},
    {'Driver': 'Campaign contact count',
     'Direction': 'Negative',
     'Plain-language explanation': 'More than 5 calls in a campaign reduces conversion probability. '
                                   'Over-contacted customers disengage.',
     'Fairness note': 'No protected attribute — actionable immediately'},
])

print('=== Explainability Framework: Key Model Drivers ===')
print(explainability.to_string(index=False))
print()
print('Fairness note: Age is a protected attribute under most financial regulations.')
print('A fairness audit across age groups is required before production deployment.')

## Step 5: Decision Framework — Threshold, Trade-offs, Sensitivity, Pilot

In [ ]:
# ── COST-BENEFIT THRESHOLD ANALYSIS ──────────────────────────────────────────
# Business assumptions (document these for executives):
COST_FP  = 8    # cost of a wasted call ($) — agent time + infrastructure
VALUE_FN = 350  # estimated annual value of a missed subscriber ($)

thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
rows = []

for t in thresholds:
    pred = (rf_prob >= t).astype(int)
    tp = int(((pred==1) & (y_test==1)).sum())
    fp = int(((pred==1) & (y_test==0)).sum())
    fn = int(((pred==0) & (y_test==1)).sum())
    called = tp + fp
    prec   = tp / called if called > 0 else 0
    rec    = tp / y_test.sum()
    net    = (tp * VALUE_FN) - (fp * COST_FP) - (fn * VALUE_FN * 0.1)
    rows.append({'Threshold': t, 'Customers called': called,
                 '% Called': f'{called/len(y_test):.1%}',
                 'Precision': f'{prec:.1%}', 'Recall': f'{rec:.1%}',
                 'False Pos': fp, 'False Neg': fn,
                 'Net Value ($)': int(net)})

thresh_df = pd.DataFrame(rows)
print(thresh_df.to_string(index=False))
print()
best_t = thresh_df.loc[thresh_df['Net Value ($)'].idxmax(), 'Threshold']
print(f'Optimal threshold by net value: {best_t}')

In [ ]:
# ── NET VALUE CHART ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
net_vals = thresh_df['Net Value ($)'].values
t_vals   = thresh_df['Threshold'].values
colors   = ['#1D9E75' if t == best_t else '#B5D4F4' for t in t_vals]
ax.bar(t_vals.astype(str), net_vals, color=colors, edgecolor='none')
ax.set_title('Net value by threshold — optimal at 0.15')
ax.set_xlabel('Threshold')
ax.set_ylabel('Estimated net value ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'${v/1000:.0f}K'))
plt.tight_layout()
plt.show()

In [ ]:
# ── SENSITIVITY / SCENARIO ANALYSIS ──────────────────────────────────────────
# Test whether the recommendation holds under different cost assumptions

pred = (rf_prob >= 0.15).astype(int)
tp = int(((pred==1) & (y_test==1)).sum())
fp = int(((pred==1) & (y_test==0)).sum())
fn = int(((pred==0) & (y_test==1)).sum())

scenarios = {
    'Base case (value=$350, call=$8)':     (350, 8),
    'Higher subscriber value ($500)':       (500, 8),
    'Lower subscriber value ($200)':        (200, 8),
    'Higher call cost ($20)':              (350, 20),
}

print('=== Sensitivity Analysis at Threshold 0.15 ===')
for name, (vfn, cfp) in scenarios.items():
    net = (tp * vfn) - (fp * cfp) - (fn * vfn * 0.1)
    print(f'  {name:<40}: ${net:>9,.0f}')

print()
print('Finding: The recommendation is robust across all tested scenarios.')
print('Even in the worst case (value=$200), the model generates positive net value.')

In [ ]:
# ── FULL PORTFOLIO PROJECTION ─────────────────────────────────────────────────
scale = 45211 / len(y_test)
rec_at_15 = tp / y_test.sum()
pct_called = (tp + fp) / len(y_test)

print('=== Full Portfolio Projection (45,211 customers) ===')
print(f'Subscribers captured per cycle : ~{int(rec_at_15 * 45211 * 0.117):,} ({rec_at_15:.1%})')
print(f'Total customers contacted      : ~{int(pct_called * 45211):,} ({pct_called:.1%})')
print(f'Customers skipped              : ~{int((1-pct_called) * 45211):,}')
print(f'Estimated annual net impact    : ~${int((tp*350 - fp*8 - fn*35)*scale):,}')
print()
print('=== Final Decision Rule ===')
print('Contact every customer with predicted probability >= 0.15.')
print('Skip all others. Review threshold monthly using live precision@top20% as KPI.')

## Step 6: Customer Segmentation — K-Means Clustering + RF Score Overlay

**Why add segmentation?**
The RF model tells us *who is likely to subscribe*, but not *why different groups behave differently*.
K-means clustering discovers natural customer groups from the data itself (unsupervised),
and overlaying the RF scores reveals which natural segments are the highest-value targets.
This combination supports more tailored campaign strategies beyond a single threshold rule.

**Method:**
- Features used: age, balance, duration, campaign contacts, previous contacts
- Optimal k selected by silhouette score (higher = more distinct clusters)
- RF subscription probability scores overlaid on each cluster

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np

# ── RELOAD ORIGINAL DATA FOR CLUSTERING ───────────────────────────────────────
# WHY: We cluster on raw, interpretable features (not the encoded model features)
# so that segment profiles are easy to explain to non-technical stakeholders.
df_orig = pd.read_csv('bank-full.csv', sep=';')
df_orig['y'] = (df_orig['y'] == 'yes').astype(int)

# Attach RF scores (trained in Step 4) to the full dataset
# Re-score the full dataset using the trained RF model
X_full = pd.concat([X_train, X_test]).sort_index()
y_full = pd.concat([y_train, y_test]).sort_index()
rf_prob_full = rf.predict_proba(X_full)[:, 1]
df_orig['rf_score'] = rf_prob_full

print(f'Dataset ready: {df_orig.shape[0]:,} rows')
print(f'RF scores attached: min={rf_prob_full.min():.3f}, max={rf_prob_full.max():.3f}')

In [ ]:
# ── FIND OPTIMAL NUMBER OF CLUSTERS ───────────────────────────────────────────
# WHY: We use silhouette score rather than just the elbow method.
# Silhouette measures how well each point fits its own cluster vs. neighbours.
# Score ranges from -1 (wrong cluster) to +1 (perfect cluster). Higher = better.

cluster_features = ['age', 'balance', 'duration', 'campaign', 'previous']
X_clust = df_orig[cluster_features].copy()

# Scale features — k-means is distance-based so scale matters
from sklearn.preprocessing import StandardScaler
scaler_c = StandardScaler()
X_clust_scaled = scaler_c.fit_transform(X_clust)

inertias, silhouettes = [], []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_clust_scaled)
    inertias.append(km.inertia_)
    # sample_size speeds up silhouette on large datasets
    sil = silhouette_score(X_clust_scaled, labels, sample_size=5000, random_state=42)
    silhouettes.append(sil)
    print(f'  k={k}: inertia={km.inertia_:,.0f} | silhouette={sil:.4f}')

best_k = list(K_range)[np.argmax(silhouettes)]
print(f'\nOptimal k by silhouette score: {best_k}')

# Plot elbow + silhouette side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(list(K_range), inertias, 'o-', color='#185FA5')
ax1.set_title('Elbow method — inertia by k')
ax1.set_xlabel('Number of clusters (k)')
ax1.set_ylabel('Inertia')
ax1.grid(alpha=0.3)

ax2.plot(list(K_range), silhouettes, 's-', color='#1D9E75')
ax2.axvline(best_k, color='#E24B4A', linestyle='--', alpha=0.7, label=f'Best k={best_k}')
ax2.set_title('Silhouette score by k (higher = better)')
ax2.set_xlabel('Number of clusters (k)')
ax2.set_ylabel('Silhouette score')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── FIT FINAL K-MEANS MODEL ───────────────────────────────────────────────────
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_orig['cluster'] = km_final.fit_predict(X_clust_scaled)

# ── NAME CLUSTERS BASED ON THEIR PROFILES ─────────────────────────────────────
# WHY: Numeric cluster labels (0,1,2...) mean nothing to executives.
# We inspect the profiles first, then assign meaningful business names.
cluster_names = {
    3: 'Engaged Converters',
    1: 'Previously Engaged',
    6: 'High-Net-Worth',
    2: 'Older Mid-Balance',
    0: 'Young Low-Balance',
    7: 'Over-Contacted',
    4: 'Campaign-Fatigued',
    5: 'Statistical Outlier',
}
priority = {
    3: 'Priority 1 - Contact first',
    1: 'Priority 2 - Re-engage',
    6: 'Priority 3 - Premium channel',
    2: 'Priority 4 - Standard campaign',
    0: 'Priority 5 - Digital first',
    7: 'Reduce - Cap at 1 contact',
    4: 'Stop - Remove from lists',
    5: 'Exclude - Statistical outlier',
}
df_orig['segment']  = df_orig['cluster'].map(cluster_names)
df_orig['priority'] = df_orig['cluster'].map(priority)

print('Cluster assignment complete.')
print(df_orig['segment'].value_counts())

In [ ]:
# ── SEGMENT PROFILES ──────────────────────────────────────────────────────────
# WHY: This table is the core output — it combines cluster characteristics
# with actual subscription rates and RF scores for executive reporting.

VALUE_PER_SUB = 350  # estimated annual value per subscriber ($)

seg_profile = df_orig.groupby(['segment', 'priority']).agg(
    customers    = ('y', 'count'),
    sub_rate     = ('y', 'mean'),
    avg_rf_score = ('rf_score', 'mean'),
    avg_age      = ('age', 'mean'),
    avg_balance  = ('balance', 'mean'),
    avg_duration = ('duration', 'mean'),
    avg_contacts = ('campaign', 'mean'),
    prior_contacts = ('previous', 'mean'),
).round(2)

seg_profile['pct_customers']   = (seg_profile['customers'] / df_orig.shape[0] * 100).round(1)
seg_profile['est_subscribers'] = (seg_profile['customers'] * seg_profile['sub_rate']).round(0).astype(int)
seg_profile['revenue_opp']     = (seg_profile['est_subscribers'] * VALUE_PER_SUB)
seg_profile = seg_profile.sort_values('sub_rate', ascending=False)

print('=== Segment Profiles ===')
display_cols = ['customers','pct_customers','sub_rate','avg_rf_score',
                'avg_age','avg_balance','avg_duration','avg_contacts','revenue_opp']
print(seg_profile[display_cols].to_string())

In [ ]:
# ── VISUALISATION: Subscription Rate + Revenue by Segment ─────────────────────
seg_plot = seg_profile.reset_index().sort_values('sub_rate', ascending=True)
seg_plot = seg_plot[seg_plot['segment'] != 'Statistical Outlier']  # exclude 1-row outlier

colors = {
    'Engaged Converters' : '#1D9E75',
    'Previously Engaged' : '#5DCAA5',
    'High-Net-Worth'     : '#185FA5',
    'Older Mid-Balance'  : '#FAC775',
    'Young Low-Balance'  : '#FAC775',
    'Over-Contacted'     : '#E24B4A',
    'Campaign-Fatigued'  : '#E24B4A',
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Chart 1: Subscription rate
bar_colors = [colors.get(s, '#888') for s in seg_plot['segment']]
ax1.barh(seg_plot['segment'], seg_plot['sub_rate'] * 100, color=bar_colors)
ax1.set_xlabel('Subscription rate (%)')
ax1.set_title('Subscription rate by segment')
ax1.grid(axis='x', alpha=0.3)
for i, (v, n) in enumerate(zip(seg_plot['sub_rate'], seg_plot['customers'])):
    ax1.text(v * 100 + 0.5, i, f'{v:.0%}  (n={n:,})', va='center', fontsize=10)

# Chart 2: Revenue opportunity
seg_rev = seg_plot.sort_values('revenue_opp', ascending=True)
rev_colors = [colors.get(s, '#888') for s in seg_rev['segment']]
ax2.barh(seg_rev['segment'], seg_rev['revenue_opp'] / 1000, color=rev_colors)
ax2.set_xlabel('Revenue opportunity ($000s)')
ax2.set_title('Est. annual revenue opportunity by segment')
ax2.grid(axis='x', alpha=0.3)
for i, v in enumerate(seg_rev['revenue_opp']):
    ax2.text(v/1000 + 2, i, f'${v/1000:.0f}K', va='center', fontsize=10)

# Legend
legend_patches = [
    mpatches.Patch(color='#1D9E75', label='Priority segments'),
    mpatches.Patch(color='#FAC775', label='Volume segments'),
    mpatches.Patch(color='#E24B4A', label='Reduce/stop contact'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, -0.05))
plt.tight_layout()
plt.show()

In [ ]:
# ── BUBBLE CHART: Sub rate vs RF score vs Customer volume ─────────────────────
# WHY: Shows at a glance which segments have high conversion, high model confidence,
# and how many customers are in each — all in one executive-ready visual.

seg_bubble = seg_profile.reset_index()
seg_bubble = seg_bubble[seg_bubble['segment'] != 'Statistical Outlier']

fig, ax = plt.subplots(figsize=(10, 6))

for _, row in seg_bubble.iterrows():
    c = colors.get(row['segment'], '#888')
    size = row['customers'] / 50   # scale bubble to customer count
    ax.scatter(row['sub_rate'] * 100, row['avg_rf_score'],
               s=size, color=c, alpha=0.75, edgecolors='white', linewidth=1.5)
    ax.annotate(row['segment'],
                xy=(row['sub_rate'] * 100, row['avg_rf_score']),
                xytext=(6, 4), textcoords='offset points', fontsize=9)

ax.set_xlabel('Subscription rate (%)', fontsize=12)
ax.set_ylabel('Avg Random Forest score', fontsize=12)
ax.set_title('Customer Segments — Subscription Rate vs. RF Score\n(bubble size = number of customers)', fontsize=13)
ax.grid(alpha=0.2)

legend_patches = [
    mpatches.Patch(color='#1D9E75', label='Priority segments'),
    mpatches.Patch(color='#185FA5', label='High-Net-Worth'),
    mpatches.Patch(color='#FAC775', label='Volume segments'),
    mpatches.Patch(color='#E24B4A', label='Reduce/stop contact'),
]
ax.legend(handles=legend_patches, fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── SEGMENT DECISION RULES ────────────────────────────────────────────────────
# Summary table ready for slides / executive memo

decision_rules = pd.DataFrame([
    {'Segment': 'Engaged Converters',  'Action': 'Contact first — senior agents',       'Sub rate': '47%', 'Revenue opp.': '$616K'},
    {'Segment': 'Previously Engaged',  'Action': 'Personalised re-engagement script',   'Sub rate': '24%', 'Revenue opp.': '$113K'},
    {'Segment': 'High-Net-Worth',      'Action': 'Relationship manager — not call ctr', 'Sub rate': '14%', 'Revenue opp.': '$49K'},
    {'Segment': 'Older Mid-Balance',   'Action': 'Standard campaign — mid-morning',     'Sub rate': '9%',  'Revenue opp.': '$422K'},
    {'Segment': 'Young Low-Balance',   'Action': 'Digital-first, then call responders', 'Sub rate': '8%',  'Revenue opp.': '$603K'},
    {'Segment': 'Over-Contacted',      'Action': 'Cap at 1 contact per cycle',          'Sub rate': '3%',  'Revenue opp.': '$38K'},
    {'Segment': 'Campaign-Fatigued',   'Action': 'Remove from lists for 6 months',      'Sub rate': '2%',  'Revenue opp.': '$4K'},
])

print('=== Segment Decision Rules ===')
print(decision_rules.to_string(index=False))
print()
print('Key insight: Engaged Converters (8% of customers) generate the same')
print('revenue opportunity as the entire Over-Contacted segment at 17x better conversion.')
print()
print('Stopping contact with Campaign-Fatigued and Over-Contacted customers frees')
print(f'~{3597+605:,} agent interactions per cycle — redirect to Priority 1-3 segments.')

## Limitations & Future Improvements


### Known Limitations

| Limitation | Impact | Mitigation |
|-----------|--------|------------|
| **Class imbalance (11.7% positive)** | Naive models predict 'no' always | Used class_weight='balanced' + AUC/recall metrics |
| **Duration leakage** | Duration unavailable pre-call | Two-stage deployment: no-duration RF for pre-call, full RF post-call |
| **Limited features** | Model misses behavioural signals (web activity, app usage) | Enrich dataset with digital channel data |
| **Logistic Regression linearity** | Misses non-linear relationships | Addressed by Random Forest in Step 4 |
| **Model drift** | Customer behaviour changes over time | Retrain quarterly; monitor precision@top20% monthly |
| **Fairness** | Age, job, marital status correlate with protected attributes | Demographic fairness audit required before go-live |

### Future Improvements

1. **Additional features** — digital engagement (clicks, app logins), transaction frequency, product holdings
2. **Advanced models** — XGBoost or LightGBM for higher AUC; calibrated probabilities for better threshold control
3. **Real-time scoring** — deploy model as an API so campaign targeting updates daily, not monthly
4. **Customer lifetime value integration** — weight the cost of a false negative by CLV, not a flat $350
5. **Multi-channel testing** — A/B test phone vs email vs in-app push per segment, feed results back into the model
6. **Fairness monitoring** — track false negative rates by age group and job type as a live dashboard metric

## Summary

| Step | Key output |
|------|-----------|
| 1. Problem framing | Threshold policy: contact customers with P(subscribe) ≥ 0.15 |
| 2. Data readiness | 45,211 rows, 52 features, stratified 80/20 split, no leakage |
| EDA | Education, marital, balance, job patterns documented |
| 3. Baseline model | Logistic Regression — AUC 0.908, Recall@20%: 76.8% |
| 4. Improved model | Random Forest — AUC 0.921, Recall@20%: 78.2% (3.6× lift) |
| Explainability | Plain-language driver table with fairness flags |
| 5. Decision framework | Threshold 0.15 maximises net value at ~$1.69M annually |
| 6. Segmentation | 8 customer segments; 3 priority, 2 volume, 2 reduce/stop |
| Limitations | 6 known limitations; 6 future improvements documented |


**Deployment recommendation:** Two-stage system — RF-without-duration for pre-call scoring;
full RF for post-call follow-up prioritisation. Retrain quarterly; monitor precision@top20% monthly.

**Key risks:** Duration leakage in deployment, model drift over time, fairness audit
required before go-live (age and job type are used as features).